In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math
import os
import random

# Reproducability Code
def seed_everything(seed=1234):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CONFIGURATION ---
SEQ_LEN = 30
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3

def create_sequences(df, feature_cols, seq_len):
    X, y = [], []
    for engine_id, group in df.groupby('id'):
        data = group[feature_cols].values
        labels = group['RUL'].values
        
        # Slide window
        for i in range(len(data) - seq_len + 1):
            X.append(data[i:i + seq_len])
            y.append(labels[i + seq_len - 1])
            
    return np.array(X), np.array(y)

def create_dataloader(X, y, shuffle=True):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, worker_init_fn=seed_worker)

# --- MODEL DEFINITION ---
class RULTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.1, seq_len=30):
        super(RULTransformer, self).__init__()
        self.input_linear = nn.Linear(input_dim, d_model)
        
        # Initialize positional encoding with a normal distribution (mean=0, std=0.02) and make it a learnable parameter
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, src):
        # src shape: [batch_size, seq_len, input_dim]
        src = self.input_linear(src) + self.pos_encoder
        output = self.transformer_encoder(src)
        # Take the output of the last time step for sequence classification/regression
        output = output[:, -1, :]
        return self.regressor(output).squeeze(1)

In [2]:
def train_and_evaluate(train_path, test_path, model_save_path, hparams=None):
    # 1. Handle hyperparameters (fallback to globals if not doing HOPT)
    if hparams is None:
        current_seq_len = SEQ_LEN
        current_lr = LEARNING_RATE
        current_num_layers = 2
    else:
        current_seq_len = hparams['seq_len']
        current_lr = hparams['lr']
        current_num_layers = hparams['num_layers']

    train_full_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    feature_cols = [c for c in train_full_df.columns if c not in ['id', 'cycle', 'RUL']]

    all_ids = train_full_df['id'].unique()
    rng = np.random.default_rng(seed=42)
    rng.shuffle(all_ids)
    split = int(len(all_ids) * 0.8)
    train_df = train_full_df[train_full_df['id'].isin(all_ids[:split])].copy()
    val_df   = train_full_df[train_full_df['id'].isin(all_ids[split:])].copy()
    test_df  = test_df.copy()

    X_train, y_train = create_sequences(train_df, feature_cols, current_seq_len)
    X_val,   y_val   = create_sequences(val_df,   feature_cols, current_seq_len)
    X_test,  y_test  = create_sequences(test_df,  feature_cols, current_seq_len)

    train_loader = create_dataloader(X_train, y_train, shuffle=True)
    val_loader   = create_dataloader(X_val,   y_val,   shuffle=False)
    test_loader  = create_dataloader(X_test,  y_test,  shuffle=False)

    model = RULTransformer(
        input_dim=len(feature_cols),
        num_layers=current_num_layers,
        seq_len=current_seq_len
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=current_lr)

    best_val_rmse = float('inf')
    final_train_rmse = 0

    for epoch in range(EPOCHS):
        model.train()
        # Accumulate sum of squared errors, then divide by N for true RMSE
        train_mse_sum = 0.0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            # loss.item() is mean MSE over batch; multiply back to get sum of squared errors
            train_mse_sum += loss.item() * batch_X.size(0)

        train_rmse = float(np.sqrt(train_mse_sum / len(train_loader.dataset)))

        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                preds = model(batch_X)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.numpy())
        val_rmse = math.sqrt(mean_squared_error(val_targets, val_preds))

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save(model.state_dict(), model_save_path)
            final_train_rmse = train_rmse

        # Print metrics every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == EPOCHS - 1:
            print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

    model.load_state_dict(torch.load(model_save_path, weights_only=True))
    model.eval()

    test_preds, test_targets = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            preds = model(batch_X)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(batch_y.numpy())
    test_rmse = math.sqrt(mean_squared_error(test_targets, test_preds))

    # Clear memory to prevent OOM across 108 loops
    import gc
    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {'Train RMSE': final_train_rmse, 'Val RMSE': best_val_rmse, 'Test RMSE': test_rmse}

# Store results by seed and dataset
all_results = {}

In [3]:
# 0. Train & Test on `linear_rul_no_norm_0` with multiple seeds
import os

BASE_DATA_DIR = '/kaggle/input/datasets/jessicalorainegan/deep-learning/data/processed-nasa-data/data_cleaning_1'

SEEDS = [42, 1234, 999]
dataset_name_0 = 'FD001_linear_rul_no_norm_0'
train_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd001.csv')
test_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd001.csv')

all_results[dataset_name_0] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_0} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_0, test_file_0, model_save_path)
    all_results[dataset_name_0][f'seed_{seed}'] = metrics


Training FD001_linear_rul_no_norm_0 with SEED=42
Epoch   1/50 | Train RMSE: 83.5306 | Val RMSE: 39.5872
Epoch   5/50 | Train RMSE: 32.5439 | Val RMSE: 31.8384
Epoch  10/50 | Train RMSE: 28.6051 | Val RMSE: 38.0637
Epoch  15/50 | Train RMSE: 23.5413 | Val RMSE: 36.3541
Epoch  20/50 | Train RMSE: 18.8300 | Val RMSE: 37.0715
Epoch  25/50 | Train RMSE: 18.0959 | Val RMSE: 35.4354
Epoch  30/50 | Train RMSE: 14.4251 | Val RMSE: 35.1524
Epoch  35/50 | Train RMSE: 13.8698 | Val RMSE: 34.9215
Epoch  40/50 | Train RMSE: 13.3804 | Val RMSE: 34.8947
Epoch  45/50 | Train RMSE: 13.0391 | Val RMSE: 35.6398
Epoch  50/50 | Train RMSE: 11.9987 | Val RMSE: 33.8883

Training FD001_linear_rul_no_norm_0 with SEED=1234
Epoch   1/50 | Train RMSE: 86.9167 | Val RMSE: 41.1785
Epoch   5/50 | Train RMSE: 33.1963 | Val RMSE: 30.4075
Epoch  10/50 | Train RMSE: 30.4300 | Val RMSE: 31.1760
Epoch  15/50 | Train RMSE: 27.9607 | Val RMSE: 33.7372
Epoch  20/50 | Train RMSE: 21.2838 | Val RMSE: 34.3017
Epoch  25/50 | Tra

In [4]:
# 1. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_1 = 'linear_rul_1'
train_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd001.csv')
test_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd001.csv')

all_results[dataset_name_1] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_1} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_1, test_file_1, model_save_path)
    all_results[dataset_name_1][f'seed_{seed}'] = metrics


Training linear_rul_1 with SEED=42
Epoch   1/50 | Train RMSE: 83.5306 | Val RMSE: 39.5872
Epoch   5/50 | Train RMSE: 32.5439 | Val RMSE: 31.8384
Epoch  10/50 | Train RMSE: 28.6051 | Val RMSE: 38.0637
Epoch  15/50 | Train RMSE: 23.5413 | Val RMSE: 36.3541
Epoch  20/50 | Train RMSE: 18.8300 | Val RMSE: 37.0715
Epoch  25/50 | Train RMSE: 18.0959 | Val RMSE: 35.4354
Epoch  30/50 | Train RMSE: 14.4251 | Val RMSE: 35.1524
Epoch  35/50 | Train RMSE: 13.8698 | Val RMSE: 34.9215
Epoch  40/50 | Train RMSE: 13.3804 | Val RMSE: 34.8947
Epoch  45/50 | Train RMSE: 13.0391 | Val RMSE: 35.6398
Epoch  50/50 | Train RMSE: 11.9987 | Val RMSE: 33.8883

Training linear_rul_1 with SEED=1234
Epoch   1/50 | Train RMSE: 86.9167 | Val RMSE: 41.1785
Epoch   5/50 | Train RMSE: 33.1963 | Val RMSE: 30.4075
Epoch  10/50 | Train RMSE: 30.4300 | Val RMSE: 31.1760
Epoch  15/50 | Train RMSE: 27.9607 | Val RMSE: 33.7372
Epoch  20/50 | Train RMSE: 21.2838 | Val RMSE: 34.3017
Epoch  25/50 | Train RMSE: 19.1495 | Val RMSE:

In [5]:
# 2. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_2 = 'FD001_piecewise_rul_2'
train_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_150_fd001.csv')
test_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_150_fd001.csv')

all_results[dataset_name_2] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_2} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_piecewise_2_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_2, test_file_2, model_save_path)
    all_results[dataset_name_2][f'seed_{seed}'] = metrics


Training FD001_piecewise_rul_2 with SEED=42
Epoch   1/50 | Train RMSE: 69.6761 | Val RMSE: 31.5769
Epoch   5/50 | Train RMSE: 19.8820 | Val RMSE: 18.4709
Epoch  10/50 | Train RMSE: 17.4745 | Val RMSE: 21.8716
Epoch  15/50 | Train RMSE: 14.9747 | Val RMSE: 21.2409
Epoch  20/50 | Train RMSE: 12.1165 | Val RMSE: 22.2608
Epoch  25/50 | Train RMSE: 11.5304 | Val RMSE: 22.1199
Epoch  30/50 | Train RMSE: 10.1818 | Val RMSE: 21.6679
Epoch  35/50 | Train RMSE: 10.0925 | Val RMSE: 23.3571
Epoch  40/50 | Train RMSE: 11.5368 | Val RMSE: 22.5402
Epoch  45/50 | Train RMSE: 9.2345 | Val RMSE: 23.5462
Epoch  50/50 | Train RMSE: 11.9030 | Val RMSE: 23.4490

Training FD001_piecewise_rul_2 with SEED=1234
Epoch   1/50 | Train RMSE: 73.7361 | Val RMSE: 29.1086
Epoch   5/50 | Train RMSE: 20.4272 | Val RMSE: 18.6218
Epoch  10/50 | Train RMSE: 18.3254 | Val RMSE: 20.2378
Epoch  15/50 | Train RMSE: 15.9417 | Val RMSE: 23.5596
Epoch  20/50 | Train RMSE: 13.5895 | Val RMSE: 22.5881
Epoch  25/50 | Train RMSE: 12

In [6]:
# 3. Train & Test on 'low_variance_1' with multiple seeds
dataset_name_3 = 'FD001_low_variance_1'

BASE_FE_DIR = '/kaggle/input/datasets/jessicalorainegan/deep-learning/data/processed-nasa-data/feature_engineering_2'

train_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/train_fd001_low_variance_1.csv')
test_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/test_fd001_low_variance_1.csv')

all_results[dataset_name_3] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_3} with SEED={seed}")
    print(f"{'='*60}")

    model_save_path = f'best_transformer_fd001_low_variance_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_3, test_file_3, model_save_path)
    all_results[dataset_name_3][f'seed_{seed}'] = metrics


Training FD001_low_variance_1 with SEED=42
Epoch   1/50 | Train RMSE: 68.8176 | Val RMSE: 25.4062
Epoch   5/50 | Train RMSE: 19.9726 | Val RMSE: 20.3673
Epoch  10/50 | Train RMSE: 18.2354 | Val RMSE: 21.3408
Epoch  15/50 | Train RMSE: 15.2471 | Val RMSE: 23.6997
Epoch  20/50 | Train RMSE: 12.6681 | Val RMSE: 21.0046
Epoch  25/50 | Train RMSE: 11.2463 | Val RMSE: 22.1514
Epoch  30/50 | Train RMSE: 10.2046 | Val RMSE: 21.6533
Epoch  35/50 | Train RMSE: 10.3127 | Val RMSE: 25.0075
Epoch  40/50 | Train RMSE: 9.6343 | Val RMSE: 24.2707
Epoch  45/50 | Train RMSE: 9.4325 | Val RMSE: 22.0067
Epoch  50/50 | Train RMSE: 9.2741 | Val RMSE: 23.5755

Training FD001_low_variance_1 with SEED=1234
Epoch   1/50 | Train RMSE: 68.2412 | Val RMSE: 29.3203
Epoch   5/50 | Train RMSE: 19.9777 | Val RMSE: 21.8789
Epoch  10/50 | Train RMSE: 18.1592 | Val RMSE: 22.5313
Epoch  15/50 | Train RMSE: 16.0244 | Val RMSE: 21.9672
Epoch  20/50 | Train RMSE: 14.0423 | Val RMSE: 23.2003
Epoch  25/50 | Train RMSE: 11.747

KeyboardInterrupt: 

In [ ]:
# Train and Test on 'FD001 Drop S14' with multiple seeds
dataset_name_7 = 'FD001_drop_s14'
train_file_7 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14.csv')
test_file_7 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14.csv') 

all_results[dataset_name_7] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_7} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_7, test_file_7, model_save_path)
    all_results[dataset_name_7][f'seed_{seed}'] = metrics

In [ ]:
# Train and Test on 'FD001 Drop S14 and S11' with multiple seeds
dataset_name_8 = 'FD001_drop_s14_s11'
train_file_8 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14_s11.csv')
test_file_8 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14_s11.csv') 

all_results[dataset_name_8] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_8} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_s11_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_8, test_file_8, model_save_path)
    all_results[dataset_name_8][f'seed_{seed}'] = metrics

In [ ]:
# Train and Test on 'FD001 Drop S14 and S12' with multiple seeds
dataset_name_9 = 'FD001_drop_s14_s12'
train_file_9 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/train_fd001_drop_s14_s12.csv')
test_file_9 = os.path.join(BASE_FE_DIR, 'high_correlation_2/manual_fd001_1/test_fd001_drop_s14_s12.csv') 

all_results[dataset_name_9] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_9} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd001_drop_s14_s12_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_9, test_file_9, model_save_path)
    all_results[dataset_name_9][f'seed_{seed}'] = metrics

In [ ]:
# 4. Train & Test on `linear_rul_no_norm_0` with multiple seeds
dataset_name_4 = 'FD002_linear_rul_no_norm_0'
train_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd002.csv')
test_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd002.csv')

all_results[dataset_name_4] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_4} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_4, test_file_4, model_save_path)
    all_results[dataset_name_4][f'seed_{seed}'] = metrics

In [ ]:
# 5. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_5 = 'FD002_linear_rul_1'
train_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd002.csv')
test_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd002.csv')

all_results[dataset_name_5] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_5} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_fd002_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_5, test_file_5, model_save_path)
    all_results[dataset_name_5][f'seed_{seed}'] = metrics

In [ ]:
# 6. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_6 = 'FD002_piecewise_rul_2'
train_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_150_fd002.csv')
test_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_150_fd002.csv')

all_results[dataset_name_6] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_6} with SEED={seed}")
    print(f"{'='*60}")
    
    # FIX (bug 3): was 'best_transformer_norm_1_seed_{seed}.pth' — same name as dataset 5!
    model_save_path = f'best_transformer_fd002_piecewise_2_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_6, test_file_6, model_save_path)
    all_results[dataset_name_6][f'seed_{seed}'] = metrics

In [ ]:
# Display detailed results and compute averages
print(f"\n{'='*80}")
print("DETAILED RESULTS BY SEED")
print(f"{'='*80}\n")

for dataset_name, seed_results in all_results.items():
    print(f"\n### {dataset_name.upper()} ###")
    df_results = pd.DataFrame(seed_results).T
    print(df_results.to_string())
    
    print(f"\n--- MEAN METRICS ---")
    mean_metrics = df_results.mean()
    for metric_name, value in mean_metrics.items():
        print(f"{metric_name}: {value:.2f}")

print(f"\n{'='*80}")
print("COMPARISON TABLE - ALL DATASETS & SEEDS")
print(f"{'='*80}\n")

# Create a comprehensive comparison table
comparison_data = []
for dataset_name, seed_results in all_results.items():
    for seed_label, metrics in seed_results.items():
        seed_num = seed_label.split('_')[1]
        comparison_data.append({
            'Dataset': dataset_name,
            'Seed': seed_num,
            'Train RMSE': metrics['Train RMSE'],
            'Val RMSE': metrics['Val RMSE'],
            'Test RMSE': metrics['Test RMSE']
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print(f"\n{'='*80}")
print("AVERAGES ACROSS ALL SEEDS")
print(f"{'='*80}\n")

# Create summary table with averages
summary_data = []
for dataset_name in all_results.keys():
    seed_dfs = [pd.DataFrame([metrics]).T for metrics in all_results[dataset_name].values()]
    avg_df = pd.concat(seed_dfs, axis=1).mean(axis=1)
    summary_data.append({
        'Dataset': dataset_name,
        'Mean Train RMSE': avg_df['Train RMSE'],
        'Mean Val RMSE': avg_df['Val RMSE'],
        'Mean Test RMSE': avg_df['Test RMSE']
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Save comparison results to CSV
print(f"\n{'='*80}")
print("SAVING RESULTS TO CSV")
print(f"{'='*80}\n")

# Save detailed results
comparison_df.to_csv('transformer_results_detailed.csv', index=False)
print("✓ Saved: transformer_results_detailed.csv")

# Save summary results
summary_df.to_csv('transformer_results_summary.csv', index=False)
print("✓ Saved: transformer_results_summary.csv")

print("\nFiles saved in the current working directory.")

# Hyperparameter Optimisation for Piecewise and Low Variance Models

This section performs grid search hyperparameter tuning for the following variables:
- `seed`
- `SEQ_LEN`
- `LEARNING_RATE`
- `num_layers`

on the `piecewise_rul_2` and `low_variance_1` models.

In [ ]:
import itertools
import shutil

# Define hyperparameter search space
SEEDS = [42, 1234, 999]
SEQ_LENS = [30, 40, 50]
LEARNING_RATES = [1e-3, 5e-4]
NUM_LAYERS = [1, 2, 4]

# Save original config to restore later
ORIG_SEQ_LEN = SEQ_LEN
ORIG_BATCH_SIZE = BATCH_SIZE
ORIG_LR = LEARNING_RATE
ORIG_EPOCHS = EPOCHS

# --- Grid search for piecewise model ---
print("\n=== HYPERPARAM OPTIMISATION: PIECEWISE MODEL ===")
piecewise_results = []
best_piecewise_val_rmse = float('inf')
best_piecewise_model_path = 'best_transformer_hopt_piecewise_FINAL.pth'
temp_path = 'temp_hopt.pth'

for seed, seq_len, lr, num_layers in itertools.product(SEEDS, SEQ_LENS, LEARNING_RATES, NUM_LAYERS):
    print(f"\n--- seed={seed}, seq_len={seq_len}, lr={lr}, num_layers={num_layers} ---")
    seed_everything(seed)
    metrics = train_and_evaluate(
        train_file_2, test_file_2, temp_path,
        hparams={'seq_len': seq_len, 'num_layers': num_layers, 'lr': lr}
    )
    # Keep only the single best model across the entire grid
    if metrics['Val RMSE'] < best_piecewise_val_rmse:
        best_piecewise_val_rmse = metrics['Val RMSE']
        shutil.copy(temp_path, best_piecewise_model_path)
        print(f"  ✓ New best piecewise model (Val RMSE: {best_piecewise_val_rmse:.4f}) → {best_piecewise_model_path}")
    if os.path.exists(temp_path):
        os.remove(temp_path)
    piecewise_results.append({
        'seed': seed, 'seq_len': seq_len, 'lr': lr, 'num_layers': num_layers,
        **metrics
    })

piecewise_df = pd.DataFrame(piecewise_results)
best_piecewise_row = piecewise_df.loc[piecewise_df['Val RMSE'].idxmin()]
print(f"\nBest piecewise config: seed={best_piecewise_row['seed']}, seq_len={best_piecewise_row['seq_len']}, lr={best_piecewise_row['lr']}, num_layers={best_piecewise_row['num_layers']}")
print(f"Best piecewise Val RMSE: {best_piecewise_val_rmse:.4f} → saved to '{best_piecewise_model_path}'")
print(piecewise_df.to_string())

# --- Grid search for low variance model ---
print("\n=== HYPERPARAM OPTIMISATION: LOW VARIANCE MODEL ===")
lowvar_results = []
best_lowvar_val_rmse = float('inf')
best_lowvar_model_path = 'best_transformer_hopt_lowvar_FINAL.pth'

for seed, seq_len, lr, num_layers in itertools.product(SEEDS, SEQ_LENS, LEARNING_RATES, NUM_LAYERS):
    print(f"\n--- seed={seed}, seq_len={seq_len}, lr={lr}, num_layers={num_layers} ---")
    seed_everything(seed)
    metrics = train_and_evaluate(
        train_file_3, test_file_3, temp_path,
        hparams={'seq_len': seq_len, 'num_layers': num_layers, 'lr': lr}
    )
    # Keep only the single best model across the entire grid
    if metrics['Val RMSE'] < best_lowvar_val_rmse:
        best_lowvar_val_rmse = metrics['Val RMSE']
        shutil.copy(temp_path, best_lowvar_model_path)
        print(f"  ✓ New best low variance model (Val RMSE: {best_lowvar_val_rmse:.4f}) → {best_lowvar_model_path}")
    if os.path.exists(temp_path):
        os.remove(temp_path)
    lowvar_results.append({
        'seed': seed, 'seq_len': seq_len, 'lr': lr, 'num_layers': num_layers,
        **metrics
    })

lowvar_df = pd.DataFrame(lowvar_results)
best_lowvar_row = lowvar_df.loc[lowvar_df['Val RMSE'].idxmin()]
print(f"\nBest low variance config: seed={best_lowvar_row['seed']}, seq_len={best_lowvar_row['seq_len']}, lr={best_lowvar_row['lr']}, num_layers={best_lowvar_row['num_layers']}")
print(f"Best low variance Val RMSE: {best_lowvar_val_rmse:.4f} → saved to '{best_lowvar_model_path}'")
print(lowvar_df.to_string())

# Restore original config
SEQ_LEN = ORIG_SEQ_LEN
LEARNING_RATE = ORIG_LR

# Save results tables
piecewise_df.to_csv('hopt_piecewise_results.csv', index=False)
lowvar_df.to_csv('hopt_lowvar_results.csv', index=False)
print("\n✓ Saved: hopt_piecewise_results.csv, hopt_lowvar_results.csv")
print(f"✓ Best models saved: '{best_piecewise_model_path}', '{best_lowvar_model_path}'")  